# MCP

In [8]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, function_tool, trace, WebSearchTool, ModelSettings
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

Let's set up our RAG database connection:

In [3]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [4]:
# This is the same code as in the rag.ipynb notebook


@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Integrate EXA Search as an MCP:

In [8]:
# Exa Search MCP code comes here:
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}",
        "timeout": 30
        },
    client_session_timeout_seconds=30,
    cache_tools_list=True,
    max_retry_attempts=1
)

await exa_search_mcp.connect()

calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the EXA web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use Exa Search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool],
    mcp_servers=[exa_search_mcp]
)

In [ ]:
calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        1) First, use the WebSearchTool to get the list of ingredients of the meal.
        2) Then, if it's about a meal, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use web search to find the exact recipe and ingredients.
    * Once you know the ingredients, always use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 8 times.
    * Don't forget to use the web search tool named WebSearchTool at least once to check for the ingredients.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()],
    #model_settings=ModelSettings(tool_choice="WebSearchTool")
)

Reference query - shouldn't use ExaSearch:

In [5]:
with trace("Nutrition Assistant with MCP - Only uses calorie_lookup_tool"):
    result = await Runner.run(
        calorie_agent_with_search,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    - Banana (1 medium, ~118 g): ~105 kcal (89 kcal per 100 g)
    - Apple (1 medium, ~182 g): ~95 kcal (52 kcal per 100 g)
    
    Total for one banana + one apple: ~200 kcal
    
    Calories per 100 g:
    - Banana: 89 kcal/100 g
    - Apple: 52 kcal/100 g
- 7 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [9]:
with trace("Nutrition Assistant with MCP "):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A typical Full English breakfast is about 1,000 calories per serving (rough estimate; portions vary).

Approximate breakdown per serving:
- Sausages (2): ~276 kcal
- Back bacon (2 rashers): ~64 kcal
- Eggs (2): ~140 kcal
- Baked beans (1 cup): ~225 kcal
- Tomatoes (1 medium): ~22 kcal
- Mushrooms (150 g): ~33 kcal
- Toast (2 slices): ~143 kcal
- Butter (for toast, 1 tbsp): ~100 kcal

Total ≈ 1,000 kcal

Note: calories vary with portion sizes and exact ingredients.


In [15]:
with trace("Nutrition Assistant with MCP "):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A traditional English (full English) breakfast typically includes: 2 eggs, 2 rashers of bacon, 2 pork sausages, baked beans, 1/2 tomato, mushrooms, and 2 slices of toast.

Estimated calories for one serving (approximate, varies by exact sizes/brands):
- Eggs (2 large): 97 kcal
- Bacon (2 rashers): ~65 kcal
- Sausages (2): ~222 kcal
- Baked beans (1/2 cup ~130 g): ~123 kcal
- Tomato (1/2 medium): ~15 kcal
- Mushrooms (1 cup sliced ~70 g): ~16 kcal
- Toast (2 slices): ~142 kcal
Total: ~680 kcal.  
Note: actual calories vary with portion sizes and brands. 


In [10]:
with trace("Nutrition Assistant with MCP "):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast?"
    )
    print(result.final_output)

A typical Full English breakfast (one serving) includes eggs, bacon, sausage, beans, tomato, mushrooms, toast and a bit of butter. With standard portions, this comes to roughly 1,110–1,150 calories per serving, though the exact total varies with brands and cooking methods. Here’s a common single-serving breakdown:

- Eggs, large: 2 eggs (about 70 kcal each) = 140 kcal. ([ams.usda.gov](https://www.ams.usda.gov/sites/default/files/media/L01%20Shell%20Egg%20Label%20Approval%2003%2001%2013.pdf?utm_source=openai))
- Back bacon: 2 rashers (about 50 g) = ~270 kcal (bacon around 541 kcal/100 g). ([calzen.ai](https://calzen.ai/en/calories-in/bacon/?utm_source=openai))
- Pork sausage: 1 large sausage (~100 g) = ~339 kcal (raw sausage ~339 kcal/100 g). ([caloriescalc.com](https://caloriescalc.com/usda-commodity-pork-sausage-bulklinkspatties-frozen-raw/?utm_source=openai))
- Baked beans: 1/2 cup (~130 g) = ~122 kcal (about 94 kcal/100 g). ([wicworks.fns.usda.gov](https://wicworks.fns.usda.gov/es/n